# AAAIM Evaluation Test

This notebook tests both single model evaluation and batch evaluation of multiple models.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv()  # defaults to .env in current directory

# Add the project root to the Python path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Import AAAIM functions
from core import annotate_model, curate_model
from core.database_search import force_clear_chromadb, get_species_recommendations_rag
from utils.evaluation import (
    evaluate_single_model,
    evaluate_models_in_folder,
    print_evaluation_results,
    compare_results,
    process_saved_llm_responses
)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# LLM configuration
# llm_model = "Llama-4-Maverick-17B-128E-Instruct-FP8"
llm_model = "Llama-3.3-70B-Instruct"
# llm_model = "meta-llama/llama-3.3-70b-instruct:free"
# llm_model = "meta-llama/llama-3.3-70b-instruct"
# llm_model = "gpt-4.1-nano"

# Evaluation parameters
max_entities_per_model = 10  # Limit entities per model for testing
num_models_to_test = 5  # Number of models to test in batch evaluation

# Entity and database configuration
entity_type = "chemical"
database = "kegg"

output_dir = "./results/"  # Output directory for results

### Test function

In [ ]:
# Test data - typical chemical species with synonyms
species_ids = ["glucose", "caffeine", "aspirin"]
synonyms_dict = {
    "glucose": ["glucose", "dextrose", "D-glucose"],
    "caffeine": ["caffeine", "1,3,7-trimethylxanthine"],
    "aspirin": ["aspirin", "acetylsalicylic acid", "ASA"]
}

print("Testing RAG-based entity linking...")
print("="*50)

try:
    # Test RAG approach
    rag_recommendations = get_species_recommendations_rag(
        species_ids=species_ids,
        synonyms_dict=synonyms_dict,
        model_type="default",
        top_k=3
    )
    
    for rec in rag_recommendations:
        print(f"\nSpecies: {rec.id}")
        print(f"Synonyms: {rec.synonyms}")
        print(f"Candidates: {rec.candidates}")
        print(f"Names: {rec.candidate_names}")
        print(f"Match scores (similarity): {rec.match_score}")
        
except Exception as e:
    print(f"RAG search failed: {e}")

## Annotating a new model with no or few existing annotations

In [ ]:
test_model_file = "190_few_anno.xml"
# Check if test model exists
if os.path.exists(test_model_file):
    print(f"✓ Test model found: {test_model_file}")
else:
    print(f"✗ Test model not found: {test_model_file}")

In [ ]:
llm_model

In [ ]:
os.environ["LLAMA_API_KEY"] = "LLM|4116740105262973|CdfOU_esexG933f332cOiyYN7Bs"
# Test with a single model
recommendations_df, metrics = annotate_model(
    model_file=test_model_file,
    llm_model=llm_model,
    method="rag",
    max_entities=max_entities_per_model,
    entity_type=entity_type,
    database=database
)

In [ ]:
(recommendations_df.head())

In [ ]:
recommendations_df

In [ ]:
metrics # luna

In [ ]:
metrics # janis

## Curate a model with existing annotations

Evaluation of a single model with existing annotations. 
Will only look at the species with existing annotations.

In [ ]:
test_model_file = "test_models/BIOMD0000000190.xml"
# Check if test model exists
if os.path.exists(test_model_file):
    print(f"✓ Test model found: {test_model_file}")
else:
    print(f"✗ Test model not found: {test_model_file}")

In [ ]:
# Test with a single model
recommendations_df, metrics = curate_model(
    model_file=test_model_file,
    llm_model=llm_model,
    method="rag",
    max_entities=max_entities_per_model,
    entity_type=entity_type,
    database=database
)

In [ ]:
metrics

In [ ]:
# janis
recommendations_df[recommendations_df['update_annotation'] != 'ignore']

In [ ]:
# luna
recommendations_df

## Test 1: Single Model Evaluation

Evaluation of a single model with existing annotations.

In [ ]:
# Test using utils evaluation function
test_model_file = "/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels/BIOMD0000001046.xml"
result_df = evaluate_single_model(
    model_file=test_model_file,
    llm_model=llm_model,
    method = 'rag',
    top_k = 3,
    # max_entities=max_entities_per_model,
    entity_type=entity_type,
    database=database,
    save_llm_results=False,
    verbose=True
)

In [ ]:
result_df

## Test 2: Batch Model Evaluation

Test the evaluation of multiple models in a directory.

In [ ]:
model_dir = "/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels"
# model_dir = "test_models"
# Check if model directory exists
if os.path.exists(model_dir):
    model_files = [f for f in os.listdir(model_dir) if f.endswith('.xml')]
    print(f"✓ Model directory found: {model_dir}")
    print(f"  - Found {len(model_files)} XML files")
    # print(f"  - Will test first {min(num_models_to_test, len(model_files))} models")

In [ ]:
# Run batch evaluation 
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model=llm_model,
    method="rag",
    top_k=3,
    entity_type=entity_type,
    database=database,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd_kegg_rag_llama-3_top3_general_prompt.csv",
    start_at=1,
    verbose=False
)

In [ ]:
# Run batch evaluation 
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-4-Maverick-17B-128E-Instruct-FP8",
    method="rag",
    top_k=3,
    entity_type=entity_type,
    database=database,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd_kegg_rag_llama-4_top3_general_prompt.csv",
    start_at=1,
    verbose=False
)

In [ ]:
# Run batch evaluation 
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-3.3-70B-Instruct",
    method="rag",
    top_k=10,
    entity_type=entity_type,
    database=database,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd_kegg_rag_llama-3_top10_prompt_adjusted.csv",
    start_at=1,
    verbose=False
)

In [ ]:
# Run batch evaluation 
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model=llm_model,
    method="rag",
    top_k=1,
    entity_type=entity_type,
    database=database,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd_kegg_rag_llama-4_top1_prompt_adjusted.csv",
    start_at=1,
    verbose=False
)

## Test 3: Evaluating previous LLM sysnonyms

In [ ]:
results_df = process_saved_llm_responses(response_folder = '/Users/luna/Desktop/CRBM/AMAS_proj/AAAIM/tests/results/Llama-3.3-70B-instruct-Meta/chemical_prompt_adjusted', 
                               model_dir = '/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels', 
                               prev_results_csv = 'results/biomd_kegg_rag_meta-llama_top3_prompt_adjusted.csv', 
                               method = "direct",
                               output_dir = './results/', 
                               output_file = 'biomd_kegg_rag_meta-llama_top1_prompt_adjusted.csv',
                               top_k = 1,
                               verbose = False)

In [ ]:
results_df = process_saved_llm_responses(response_folder = '/Users/luna/Desktop/CRBM/AMAS_proj/AAAIM/tests/results/llama-4-maverick-17b-128e-instruct-fp8/chemical_prompt_adjusted', 
                               model_dir = '/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels', 
                               prev_results_csv = 'results/biomd_kegg_rag_llama-4_top3_prompt_adjusted.csv', 
                               method = "rag",
                               output_dir = './results/', 
                               output_file = 'biomd_kegg_rag_llama-4_top3_prompt_adjusted_rerun.csv',
                               top_k = 3,
                               verbose = False)

## Statistics

In [ ]:
print_evaluation_results("results/biomd_kegg_rag_llama-4_top3_general_prompt.csv")

In [ ]:
print_evaluation_results("results/biomd_kegg_rag_llama-4_top10_prompt_adjusted.csv")

In [ ]:
print_evaluation_results("results/biomd_kegg_rag_llama-3_top3_prompt_adjusted.csv")

In [ ]:
print_evaluation_results("results/biomd_kegg_rag_meta-llama_top3_prompt_adjusted.csv")

In [ ]:
compare_results(
    'results/biomd_kegg_rag_meta-llama_top1_prompt_adjusted.csv',
    'results/biomd_kegg_rag_meta-llama_top3_prompt_adjusted.csv',
    'results/biomd_kegg_rag_meta-llama_top5_prompt_adjusted.csv',
    'results/biomd_kegg_rag_meta-llama_top10_prompt_adjusted.csv',
    'results/biomd_kegg_rag_llama-4_top3_prompt_adjusted.csv',
    'results/biomd_kegg_rag_llama-4_top10_prompt_adjusted.csv',
    'results/biomd_kegg_direct_llama-4_top3_prompt_adjusted.csv'
)

In [ ]:
print_evaluation_results("results/biomd_kegg_rag_llama-4_top3_prompt_adjusted.csv")

In [ ]:
df1 = pd.read_csv('results/biomd_kegg_rag_llama-4_top3_repeat2.csv')
df2 = pd.read_csv('results/biomd_kegg_rag_llama-4_top3_repeat2_rerun.csv')

# Merge on model and species_id, suffixing accuracy columns
merged = pd.merge(
    df1[['model', 'species_id', 'accuracy']],
    df2[['model', 'species_id', 'accuracy']],
    on=['model', 'species_id'],
    suffixes=('_1', '_2')
)

# Find rows where accuracy differs (including NaN vs value)
diffs = merged[
    (merged['accuracy_1'] != merged['accuracy_2']) |
    (merged['accuracy_1'].isnull() != merged['accuracy_2'].isnull())
]

# Print the differing model/species_id pairs
for _, row in diffs.iterrows():
    print(f"model: {row['model']}, species_id: {row['species_id']}, accuracy_1: {row['accuracy_1']}, accuracy_2: {row['accuracy_2']}")

In [ ]:
compare_results('results/biomd_kegg_rag_llama-4_top3.csv','results/biomd_kegg_rag_llama-4_top3_rerun.csv', 'results/biomd_kegg_rag_llama-4_top3_repeat2.csv', 'results/biomd_kegg_rag_llama-4_top3_repeat2_rerun.csv')

In [ ]:
recommendations_df[recommendations_df['id']=='Nb']

In [ ]:
print_evaluation_results('results/biomd_kegg_rag_llama-4_top3.csv') 

In [ ]:
print_evaluation_results('results/biomd_kegg_rag_meta-llama_default.csv')

In [ ]:
print_evaluation_results('results/biomd_kegg_rag_gpt-4.1-nano_default.csv')

In [ ]:
gpt_df_filtered.to_csv('results/biomd_kegg_rag_gpt-4.1-nano_default_filtered.csv', index=False)

In [ ]:
prev_df = pd.read_csv('results/biomd_kegg_direct_llama_plain_nosymbols.csv')
prev_models = set(prev_df['model'].unique())
new_df = pd.read_csv('results/biomd_kegg_rag_meta-llama_top3.csv')
new_models = set(new_df['model'].unique())
new_models = new_models - prev_models
new_models

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
results_llama = pd.read_csv('results/biomd_kegg_rag_meta-llama_default.csv')
# Group models by number of species and calculate accuracy statistics
model_species_count = results_llama.groupby('model').size().reset_index(name='species_count')
model_accuracy = results_llama.groupby('model')['accuracy'].mean().reset_index()

# Merge the two dataframes
model_stats = pd.merge(model_species_count, model_accuracy, on='model')

# Create bins for species count to make the box plot more readable
bins = [0, 5, 10, 20, 50, 100, 1000]
labels = ['1-5', '6-10', '11-20', '21-50', '51-100', '100+']
model_stats['species_count_bin'] = pd.cut(model_stats['species_count'], bins=bins, labels=labels)

# Count number of models in each bin
bin_counts = model_stats['species_count_bin'].value_counts().reindex(labels, fill_value=0)

# Create the box plot
plt.figure(figsize=(6, 5))
ax = sns.boxplot(x='species_count_bin', y='accuracy', data=model_stats, medianprops={"color": "yellow", "linewidth": 1})

# Add counts above each box
for i, label in enumerate(labels):
    count = bin_counts[label]
    ax.text(i, 1.01, f'n={count}', ha='center', va='bottom', fontsize=10, color='black', fontweight='bold')

plt.title('Model accuracy by number of species in a model')
plt.xlabel('Number of species in Model')
plt.ylabel('Average model accuracy')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
# plt.savefig(os.path.join(output_dir, 'accuracy_by_species_count.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# fix display name handling
ref_df = pd.read_csv('/Users/luna/Desktop/CRBM/AMAS_proj/Results/biomd_species_accuracy_AMAS.csv')
df = pd.read_csv('/Users/luna/Desktop/CRBM/AMAS_proj/AAAIM/tests/results/biomd_kegg_rag_meta-llama_default.csv')

# Replace 'display_name' in df with 'species_name' from ref_df using 'model' and 'species_id' as keys
df = df.merge(ref_df[['model', 'species_id', 'species_name']], on=['model', 'species_id'], how='left', suffixes=('', '_ref'))
df['display_name'] = df['species_name']
df = df.drop(columns=['species_name'])
df.to_csv('/Users/luna/Desktop/CRBM/AMAS_proj/AAAIM/tests/results/biomd_kegg_rag_meta-llama_default_display_names.csv', index=False)

# LLM synergy

In [ ]:
from utils.evaluation import evaluate_llm_synergy

# Compare multiple LLM results
synergy_df = evaluate_llm_synergy(
    'results/biomd_kegg_rag_meta-llama_top3.csv',
    'results/biomd_kegg_rag_gpt-4.1-nano_default.csv',
    'results/biomd_kegg_rag_llama-4_top3.csv',
    output_file='llm_synergy_analysis.csv'
)

# Save filtered results

In [ ]:
print_evaluation_results("/Users/luna/Desktop/CRBM/AMAS_proj/AAAIM/tests/results/biomd_kegg_rag_llama-3_top10_prompt_adjusted.csv", ref_results_csv=None)

In [ ]:
results_xlsx = "/Users/luna/Desktop/CRBM/AMAS_proj/AAAIM/tests/results/biomd_kegg_rag_llama-3_top10_prompt_adjusted_review.xlsx"
ref_results_csv = "/Users/luna/Desktop/CRBM/AMAS_proj/Results/biomd_species_accuracy_AMAS.csv"
df = pd.read_excel(results_xlsx)

# Filter by reference results
ref_df = pd.read_csv(ref_results_csv)
ref_pairs = set(zip(ref_df['model'], ref_df['species_id']))
mask = df.apply(lambda row: (row['model'], row['species_id']) in ref_pairs, axis=1)
df = df[mask]

print(f"Filtered results to {len(df)} entries that exist in reference: {ref_results_csv}")

In [ ]:
df.to_excel('results/biomd_kegg_rag_llama-3_top10_prompt_adjusted_is+isVersionOf_review.xlsx', index=False)